In [27]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupShuffleSplit
from scipy.sparse import hstack
from sklearn.metrics import classification_report
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    classification_report,
)
import plotly.graph_objects as go
import sys
sys.path.append('..')
from funs import *

In [28]:
data = get_df()
data

,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target,Nationality,Dropout
0,Single,2nd phase - general contingent,5,Animation and Multimedia Design,Daytime,Secondary education,122.0,Basic Education 3rd Cycle (9th/10th/11th Year)...,Other - 11th Year of Schooling,"Personal Services, Security and Safety Workers...",...,0,0,0.000000,0,10.8,1.4,1.74,Dropout,Portugese,True
1,Single,International student (bachelor),1,Tourism,Daytime,Secondary education,160.0,Secondary Education - 12th Year of Schooling o...,Higher Education - Degree,Intermediate Level Technicians and Professions,...,6,6,13.666667,0,13.9,-0.3,0.79,Graduate,Portugese,False
2,Single,1st Phase General Contingent,5,Communication Design,Daytime,Secondary education,122.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,Unskilled Workers,...,0,0,0.000000,0,10.8,1.4,1.74,Dropout,Portugese,True
3,Single,2nd phase - general contingent,2,Journalism and Communication,Daytime,Secondary education,122.0,Basic Education 2nd Cycle (6th/7th/8th Year) o...,Basic education 1st cycle (4th/5th year) or eq...,"Personal Services, Security and Safety Workers...",...,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate,Portugese,False
4,Married,Over 23 years old,1,Social Service (evening attendance),Evening,Secondary education,100.0,Basic education 1st cycle (4th/5th year) or eq...,Basic Education 2nd Cycle (6th/7th/8th Year) o...,Unskilled Workers,...,6,6,13.000000,0,13.9,-0.3,0.79,Graduate,Portugese,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,Single,1st Phase General Contingent,6,Journalism and Communication,Daytime,Secondary education,125.0,Secondary Education - 12th Year of Schooling o...,Secondary Education - 12th Year of Schooling o...,"Personal Services, Security and Safety Workers...",...,8,5,12.666667,0,15.5,2.8,-4.06,Graduate,Portugese,False
4420,Single,1st Phase General Contingent,2,Journalism and Communication,Daytime,Secondary education,120.0,Secondary Education - 12th Year of Schooling o...,Secondary Education - 12th Year of Schooling o...,Unskilled Workers,...,6,2,11.000000,0,11.1,0.6,2.02,Dropout,Russian,True
4421,Single,1st Phase General Contingent,1,Nursing,Daytime,Secondary education,154.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,Unskilled Workers,...,9,1,13.500000,0,13.9,-0.3,0.79,Dropout,Portugese,True
4422,Single,1st Phase General Contingent,1,Management,Daytime,Secondary education,180.0,Basic education 1st cycle (4th/5th year) or eq...,Basic education 1st cycle (4th/5th year) or eq...,"Skilled Workers in Industry, Construction and ...",...,6,5,12.000000,0,9.4,-0.8,-3.12,Graduate,Portugese,False


Strong correlation
- Daytime/evening attendance
- Debtor
- Tuition fees up to date
- Gender
- Scholarship holder
- Curricular units 1st sem (grade)
- Curricular units 2nd sem (grade)

Weak correlation
- Marital status
- Application mode
- Course
- Previous qualification
- Mother's qualification
- Father's qualification
- Mother's occupation
- Father's occupation
- Displaced
- Educational special needs
- International
- Curricular units 1st sem (approved)
- Curricular units 2nd sem (enrolled)
- Curricular units 2nd sem (evaluations)
- Curricular units 2nd sem (approved)
- Inflation rate
- GDP

No correlation
- Application order
- Previous qualification grade
- Nationality
- Admission grade
- Unemployment rate
- Curricular units 1st sem (credited)
- Curricular units 1st sem (enrolled)
- Curricular units 1st sem (evaluations)
- Curricular units 2nd sem (credited)
- Curricular units 2nd sem (without evaluations)

In [29]:
categorical = [
    'Daytime/evening attendance', 
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder'
]
numerical = ['Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)']
target = ['Dropout']

In [30]:
TEST_SIZE = 0.2

X = data[categorical + numerical]
y = (data['Dropout']).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y
)

In [31]:
N_ESTIMATORS = 200
RANDOM_STATE = 42

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        # ('num', StandardScaler(), numerical)
    ]
)

model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

pipe = Pipeline([
    ("preprocessor", preprocessor), 
    ("rf", model)
])

pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [32]:
predictions = pipe.predict(X_test)
probabilities = pipe.predict_proba(X_test)

In [33]:
def plot_roc_curves(y_true, scores_dict, out_html="roc_curve.html"):
    fig = go.Figure()
    for name, y_score in scores_dict.items():
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{name} (AUC={auc:.3f})"))
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines",
                             name="Chance", line=dict(dash="dash")))
    fig.update_layout(
        title="ROC Curve Comparison",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        template="plotly_white",
        legend_title_text=None,
        width=900, height=550
    )
    fig.write_html(out_html, include_plotlyjs="cdn")

def plot_pr_curves(y_true, scores_dict, out_html="pr_curve.html"):
    fig = go.Figure()
    pos_rate = (y_true.sum() / len(y_true)) if len(y_true) else 0.0
    for name, y_score in scores_dict.items():
        precision, recall, _ = precision_recall_curve(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines",
                                 name=f"{name} (AP={ap:.3f})"))
    fig.add_trace(go.Scatter(x=[0,1], y=[pos_rate, pos_rate], mode="lines",
                             name=f"Baseline (pos rate={float(pos_rate):.3f})",
                             line=dict(dash="dash")))
    fig.update_layout(
        title="Precision–Recall Curve",
        xaxis_title="Recall",
        yaxis_title="Precision",
        template="plotly_white",
        legend_title_text=None,
        width=900, height=550
    )
    fig.write_html(out_html, include_plotlyjs="cdn")

plot_roc_curves(y_test, {"RandomForest": probabilities[:, 1]}, out_html="rf_roc_curve.html")
plot_pr_curves(y_test, {"RandomForest": probabilities[:, 1]}, out_html="rf_pr_curve.html")

In [34]:
y_preds = pipe.predict(X_test)
print(classification_report(y_test, y_preds))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_preds))

              precision    recall  f1-score   support

           0       0.76      0.98      0.85       601
           1       0.88      0.34      0.49       284

    accuracy                           0.77       885
   macro avg       0.82      0.66      0.67       885
weighted avg       0.80      0.77      0.74       885

Confusion Matrix:
[[588  13]
 [188  96]]
